# M2.2 · Point-in-time feature/label joins

_Curriculum · Domain 0 · ML Foundations · Feature engineering & leakage_

**Freeze features at prediction time, then let the label window unfold after that moment.**

We will build a tiny ads event log, create a correct as-of feature with $\text{timestamp}(e_j)<t_i$, and compare it with a leaky feature that counts clicks inside the label window. _Save a copy to your Drive (File -> Save a copy in Drive) to keep your edits._

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

## Build impressions and delayed click outcomes

Each row is an impression at freeze time $t_i$. The label asks whether a click arrives in $[t_i,t_i+\Delta)$, with $\Delta=10$ minutes.

In [ ]:
n = 240
campaigns = np.array(["A", "B", "C"])
impression_times = pd.date_range("2026-01-01 09:00", periods=n, freq="min")
campaign = rng.choice(campaigns, size=n, p=[0.45, 0.35, 0.20])
label = (rng.random(n) < 0.24).astype(int)

impressions = pd.DataFrame({"impression_id": np.arange(n), "campaign": campaign, "t": impression_times, "label": label})

print(impressions.head())

## Create the click event log

Positive labels create a future click inside the 10-minute window. We also add background clicks before and between impressions so the as-of feature has real history.

In [ ]:
positive_rows = impressions[impressions["label"] == 1].copy()
delays = rng.integers(1, 10, size=len(positive_rows))
positive_clicks = pd.DataFrame({"campaign": positive_rows["campaign"].to_numpy(), "click_time": positive_rows["t"].to_numpy() + pd.to_timedelta(delays, unit="min"), "source": "label_window"})

background_n = 90
background_offsets = rng.integers(-80, n, size=background_n)
background_clicks = pd.DataFrame({"campaign": rng.choice(campaigns, size=background_n), "click_time": pd.Timestamp("2026-01-01 09:00") + pd.to_timedelta(background_offsets, unit="min"), "source": "background"})

clicks = pd.concat([positive_clicks, background_clicks], ignore_index=True)
clicks = clicks.sort_values("click_time").reset_index(drop=True)

print(clicks.head())
print(clicks["source"].value_counts())

## Correct as-of feature

For each campaign, compute a running click count and use `merge_asof` with `allow_exact_matches=False`. That enforces the strict rule $\text{click\_time}<t_i$.

In [ ]:
click_counts = clicks.sort_values(["campaign", "click_time"]).copy()
click_counts["prior_clicks_after_event"] = click_counts.groupby("campaign").cumcount() + 1

pieces = []
for name in campaigns:
    left = impressions[impressions["campaign"] == name].sort_values("t")
    right = click_counts[click_counts["campaign"] == name].sort_values("click_time")
    joined = pd.merge_asof(left, right[["click_time", "prior_clicks_after_event"]], left_on="t", right_on="click_time", direction="backward", allow_exact_matches=False)
    pieces.append(joined)

asof = pd.concat(pieces, ignore_index=True)
asof["prior_clicks"] = asof["prior_clicks_after_event"].fillna(0).astype(int)
asof = asof.sort_values("impression_id").reset_index(drop=True)

print(asof[["impression_id", "campaign", "t", "prior_clicks", "label"]].head())

## A naive feature that leaks

Now count clicks from the same campaign through $t_i+10$ minutes. This feature includes the exact outcome window used to define the label.

In [ ]:
def count_through_window(row):
    same_campaign = clicks["campaign"] == row["campaign"]
    before_window_end = clicks["click_time"] < row["t"] + pd.Timedelta(minutes=10)
    return int((same_campaign & before_window_end).sum())

asof["leaky_clicks_through_window"] = asof.apply(count_through_window, axis=1)
asof["leaky_increment"] = asof["leaky_clicks_through_window"] - asof["prior_clicks"]

print(asof[["impression_id", "prior_clicks", "leaky_clicks_through_window", "leaky_increment", "label"]].head(10))

## Measure the leakage signal

A correct historical count can be useful, but it should not contain the label window itself. The leaky increment should correlate much more strongly with $y_i$ because it includes future clicks.

In [ ]:
corr_prior = asof["prior_clicks"].corr(asof["label"])
corr_leaky_increment = asof["leaky_increment"].corr(asof["label"])

print("corr(prior_clicks, label):", round(float(corr_prior), 3))
print("corr(leaky_increment, label):", round(float(corr_leaky_increment), 3))

assert corr_leaky_increment > 0.35
assert abs(corr_prior) < 0.25
assert corr_leaky_increment > corr_prior + 0.35

## Delayed labels and censored rows

If the data extract time is before $t_i+\Delta$, the row is not a safe negative. It is censored because a positive click could still arrive.

In [ ]:
extract_time = impressions["t"].max() + pd.Timedelta(minutes=5)
asof["window_closed"] = asof["t"] + pd.Timedelta(minutes=10) <= extract_time
closed_count = int(asof["window_closed"].sum())
censored_count = int((~asof["window_closed"]).sum())

print("closed rows:", closed_count)
print("censored rows:", censored_count)

assert censored_count == 5

## Practice

Try each in the empty cell below.

1. Change $\Delta$ from 10 minutes to 5 minutes and rebuild the leaky count.
2. Add a same-time click and confirm that `allow_exact_matches=False` excludes it from the feature.
3. Replace the final random split you would normally use with a chronological split, then compare the feature distributions.

In [ ]:
# Your turn:
